# 02 · Feature engineering

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| **2. Feature engineering** | **this notebook** |
| 3. Model training | `03_model_training.ipynb` |
| 4. Evaluation | `04_evaluation.ipynb` |

Everything here calls `src/aerocast/features.py`, which is the same code the pipeline script uses.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from aerocast import config, data, eda, features, leakage, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)

## 1. Decision time and the past-only rule

> A prediction for flights on day **D** is made at the **end of day D-1**.

| Information | Allowed? | Why |
|---|---|---|
| Timetable for day D (time, carrier, route, distance, assigned aircraft) | ✅ | published in advance |
| Number of flights scheduled at the same airport and hour on D | ✅ | computed from the timetable, **including flights later cancelled** |
| Outcomes (delays, cancellations) on days ≤ D-1 | ✅ | already happened |
| Outcomes on day D (including earlier flights the same day) | ❌ | not known at the end of D-1 |

The rule is implemented once, in `features.rolling_past`: aggregate outcomes to **(key, date)** totals → roll a trailing window over dates → **shift by one day** → look up each flight's (key, D).

## 2. Build all features

Features are built on the **full** schedule (including cancelled flights), so congestion and cancellation history are correct. Modelling rows are filtered afterwards.

In [2]:
df = data.add_target(data.load_raw())
X, groups = features.build_features(df)
print(X.shape)
pd.Series({g: len(c) for g, c in groups.items()}, name="n_features").to_frame()

(336776, 45)


,n_features
categorical,3
schedule,10
congestion,7
history_entity,18
history_day,7


In [3]:
# Restrict the descriptive views below to the TRAINING window (Jan 8 - Aug 31)
train = (df["date"] > config.WARMUP_END) & (df["date"] <= config.SPLITS["train"][1]) & df["is_delayed"].notna()
print(f"training rows: {train.sum():,}")

training rows: 211,684


## 3. Feature groups

### 3.1 Identifiers (`categorical`)
`carrier`, `origin`, `dest` as pandas categoricals. XGBoost splits on them natively. Logistic regression one-hot encodes them and groups categories with fewer than 50 flights.

In [4]:
X.loc[train, groups["categorical"]].describe()

,carrier,origin,dest
count,211684,211684,211684
unique,16,3,102
top,UA,EWR,ATL
freq,37403,76559,10924


### 3.2 Schedule & calendar (`schedule`)
* **Cyclic encodings** of departure time, arrival time and day of week: sin/cos of 2π·value/period. Without this, 23:59 and 00:01 would look maximally far apart to a linear model.
* `dep_hour` (raw): lets a linear model capture the steady rise in delays through the day.
* `days_to_holiday` (federal holidays, capped at 14) and `log_distance`.
* **No month feature.** See §6.

In [5]:
fig = plots.feature_histograms(X.loc[train], groups["schedule"], "Schedule & calendar features (train)")
plots.save(fig, "feat_schedule_dist"); fig

<Figure size 1240x630 with 12 Axes>

### 3.3 Scheduled congestion & aircraft rotation (`congestion`)
These are counts over the **published timetable** of day D: flights in the same origin-hour, the ±1-hour bank, the day; the destination's daily volume; the carrier's volume at the origin. `tail_leg_idx` / `tail_legs_day` give the position of this departure in the aircraft's NYC departures that day. A later leg inherits upstream delay.

In [6]:
fig = plots.feature_histograms(X.loc[train], groups["congestion"], "Congestion & rotation features (train)")
plots.save(fig, "feat_congestion_dist"); fig

<Figure size 1240x420 with 8 Axes>

Note: an aircraft makes only about 1.3 NYC departures a day, because the data only contains *departures from NYC*. Most `tail_leg_idx` values are therefore 1. This is a limitation of the dataset, not of the method.

### 3.4 Entity-level history (`history_entity`)
Smoothed past delay rates plus log flight-volume for: carrier (7 d, 28 d), destination, route, carrier-route, origin × hour, flight number, tail number (28 d). The volume (`hist_logn_*`) tells the model how much to trust each rate.

In [7]:
fig = plots.feature_histograms(X.loc[train], groups["history_entity"], "Entity-level history features (train)")
plots.save(fig, "feat_history_entity_dist"); fig

<Figure size 1240x1050 with 20 Axes>

### 3.5 Day-level history (`history_day`)
Yesterday's network-wide and per-origin delay rate, the 7-day network rate, yesterday's cancellation rates, and yesterday's mean arrival delay at the origin.

In [8]:
fig = plots.feature_histograms(X.loc[train], groups["history_day"], "Day-level history features (train)", ncols=4)
plots.save(fig, "feat_history_day_dist"); fig

<Figure size 1240x420 with 8 Axes>

## 4. Worked example: shift-by-one-day and smoothing

`features.history_trace` reruns the **real** `rolling_past` and `smoothed_rate` functions and reports them day by day for one key. Example: route **EWR → CAE** (a thin route, about 9 flights in January), 28-day window.

The smoothed rate is a beta-binomial (Bayesian) shrinkage estimate:

$$\hat r_{k,D} = \frac{\text{late}_{k,[D-28,\,D-1]} + m\,\pi_D}{\text{flights}_{k,[D-28,\,D-1]} + m}, \qquad m = 20$$

where $\pi_D$ is the network-wide delay rate over the 28 days before D (also past-only).

In [9]:
trace = features.history_trace(df, ("origin", "dest"), ("EWR", "CAE"), window=28)
trace.head(12)

,flights_today,late_today,hist_flights,hist_late,network_prior,hist_rate
date,,,,,,
2013-01-03,1,1.0,0.0,0.0,0.3076,0.3076
2013-01-06,1,1.0,1.0,1.0,0.2444,0.2804
2013-01-10,1,0.0,2.0,2.0,0.2034,0.2758
2013-01-13,1,1.0,3.0,2.0,0.1782,0.2419
2013-01-17,1,1.0,4.0,3.0,0.2128,0.3023
2013-01-20,1,1.0,5.0,4.0,0.2055,0.3244
2013-01-24,1,1.0,6.0,5.0,0.2121,0.3554
2013-01-27,1,0.0,7.0,6.0,0.2242,0.3883
2013-01-31,0,0.0,8.0,6.0,0.2207,0.3720


How to read the table (`*_today` = what happens on day D; `hist_*` = what the feature can see):
* **3 Jan**, the route's first flight: no history (`hist_flights = 0`), so `hist_rate` equals the network prior exactly.
* **6 Jan**: the history contains exactly the 3 Jan flight (1 flight, 1 late). Today's outcome is **not** included. It only appears in the next flying day's history (10 Jan: 2 flights).
* **Shrinkage:** after 1 late flight out of 1, the rate is **0.28**, not 1.0. With m = 20 pseudo-flights, a thin route only moves gradually away from the network level.
* **31 Jan**: `flights_today = 0` because that flight was cancelled. It's on the schedule but has no outcome, so it adds nothing to the history.
* **3 Feb**: `hist_flights` drops from 8 to 7, because 3 Jan has left the 28-day window [6 Jan, 2 Feb].
* For **1 Jan** there is no past at all (the prior itself is NaN). That's why the first week is excluded from training (`config.WARMUP_END`).

## 5. Leakage check (inline)

This is the same check that `tests/test_no_leakage.py` runs (`aerocast.leakage`):
1. Overwrite **every** post-departure column (delays, departure times, cancellations) from a cutoff date onward with random values.
2. Rebuild **all** features.
3. Assert that every numeric feature on or before the cutoff is **bit-for-bit unchanged**. The cutoff day is included, because same-day outcomes aren't known at D-1.
4. Assert that features **after** the cutoff did change. Otherwise the check would pass trivially.

In [10]:
check = leakage.future_dependence_check(df, "2013-06-15", X=X)
assert check["changed_up_to_cutoff"] == 0, "LEAK: a feature on/before the cutoff depends on later outcomes"
assert check["changed_after_cutoff"] > 0, "vacuous check: scrambling changed nothing"
pd.Series({k: v for k, v in check.items() if k != "changed_columns_after"}).to_frame("value")

,value
cutoff,2013-06-15
rows_up_to_cutoff,151859
changed_up_to_cutoff,0
rows_after_cutoff,184917
changed_after_cutoff,4165999


In [11]:
pd.Series(check["changed_columns_after"], name="columns that (correctly) react to scrambled past outcomes").to_frame()

,columns that (correctly) react to scrambled past outcomes
0,hist_cancel_network_1d
1,hist_cancel_origin_1d
2,hist_logn_carrier_28d
3,hist_logn_carrier_7d
4,hist_logn_carrier_flight_28d
5,hist_logn_carrier_origin_dest_28d
6,hist_logn_dest_28d
7,hist_logn_origin_1d
8,hist_logn_origin_7d
9,hist_logn_origin_dest_28d


**Passed.** Zero cells changed up to the cutoff, and only the history columns react to the scrambled outcomes. Schedule and congestion features don't change at all, as expected, because they don't depend on outcomes.

## 6. Why there is no month feature

In [12]:
splits = data.assign_split(df)
pd.crosstab(df["month"], splits).loc[:, list(config.SPLITS)]

col_0,train,calib,test
month,,,
1,27004,0,0
2,24951,0,0
3,28834,0,0
4,28330,0,0
5,28796,0,0
6,28243,0,0
7,29425,0,0
8,29327,0,0
9,0,27574,0


* Training covers **months 1–8 only**. Months 9–12 never appear in training, so any learned month effect would be an extrapolation. A tree model would simply treat December like August (the nearest value it has seen).
* Even within training, each month is a **single realisation** (one June, one July). A month coefficient would just memorise that month's weather, not a reusable seasonal pattern. With several years of data it would be legitimate.
* In an early version the cyclic month features were included. Removing them **improved** the held-out XGBoost PR-AUC (0.377 → 0.382), consistent with this argument.
* Seasonality still reaches the model through the **trailing 7/28-day history rates**, which adapt as the season changes.

## 7. Why day-level history is a separate group

In [13]:
tr = df.loc[train]
dv = eda.distinct_values(X.loc[train], groups["history_day"] + ["hist_rate_carrier_origin_dest_28d", "hist_rate_tailnum_28d"], tr["date"])
dv

,distinct_values,values_per_day,rows,days
hist_rate_network_1d,236,1.0000,211684,236
hist_rate_network_7d,236,1.0000,211684,236
hist_rate_origin_1d,708,3.0000,211684,236
hist_logn_origin_1d,158,2.9788,211684,236
hist_cancel_network_1d,227,1.0000,211684,236
hist_cancel_origin_1d,462,2.8814,211684,236
hist_meandelay_origin_1d,708,3.0000,211684,236
hist_rate_carrier_origin_dest_28d,56209,238.1737,211684,236
hist_rate_tailnum_28d,49628,210.2924,211684,236


`hist_rate_network_1d` takes **exactly one value per training day** (236 values over 236 days), and the per-origin variants about three per day. They are shared by roughly 300–900 flights each. Consequences:
* The **effective sample size** for these features is about 236 days, not 211k flights.
* A tree can use their exact values as a **date identifier** and memorise which training days were bad, which is overfitting that doesn't transfer.
* Entity-level features, by contrast, take around 50,000 distinct values.

Keeping them as a separate group (`history_day`) lets the ablation in notebook 04 measure their contribution in isolation. In rolling CV they help (bursty summer storms), but on the Nov–Dec test window they add nothing.

## 8. History features vs target (training window only)

Correlations are computed on **training rows only**, so no information from later periods is used to judge features.

In [14]:
hist_cols = [c for c in groups["history_entity"] + groups["history_day"] if "rate" in c or "mean" in c]
corr = X.loc[train, hist_cols].corrwith(df.loc[train, "is_delayed"]).sort_values()
fig = plots.hbar(corr.values, corr.index.tolist(), "Correlation with is_delayed (train window)", "Pearson r")
plots.save(fig, "feat_history_corr"); fig

<Figure size 550x444 with 1 Axes>

Every history rate is positively correlated with the target. The two strongest single predictors are the **finest-grained** keys: the specific flight number (r ≈ 0.23) and the origin × departure-hour slot (r ≈ 0.23). Coarse keys (network, origin over 7 days, destination) are weakest. Day-level shock features fall in the middle (≈ 0.12–0.14).

Individually the correlations are modest (r ≈ 0.05–0.23), which is typical of a day-ahead problem without weather data, and it explains why the models in notebooks 03–04 reach a moderate ROC-AUC of about 0.67.